In [1]:
!pip install langchain
!pip install groq
!pip install pymupdf
!pip install langchain_groq
!pip install dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import langchain
from langchain_core.tools import tool

In [3]:
lease_contract = "C:\\Users\\urja tendolkar\\Desktop\\projects\\ClauseIQ\\training_docs\\lease_doc(contradict).pdf"
sell_contract = "C:\\Users\\urja tendolkar\\Desktop\\projects\\ClauseIQ\\training_docs\\sell_doc.pdf"

### Pdf content extraction

In [4]:
import re
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import pymupdf as fitz

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_num, page in enumerate(doc):

        # Extract page text
        words = page.get_text("words", sort = True)
        
        lines = []
        current_line = []
        current_y = None

        for word in words:
            x0, y0, x1, y1, text, block_no, line_no, word_no = word

            # New line if vertical position changes
            if current_y is None or abs(y0 - current_y) <= 3:
                current_line.append(word)
            else:
                current_line.sort(key=lambda w: w[0])
                lines.append(" ".join(w[4] for w in current_line))
                current_line = [word]

            current_y = y0

        if current_line:
            current_line.sort(key=lambda w: w[0])
            lines.append(" ".join(w[4] for w in current_line))

        pages.append({
            "page": page_num + 1,
            "text": "\n".join(lines)
        })


    doc.close()

    return pages
output = extract_pdf_text(lease_contract)
text = "\n".join(page["text"] for page in output)
# buy_text, buy_annotations = extract_pdf_text(sell_contract)
print(f"Extracted text: {text}") 
# print(f"Extracted text: {buy_text[:1000]}...") 
# Print first 100 bytes for preview


Extracted text: Final model draft agreed to by MoUD and DoLR
Note: -This is a model draft and may be customized according to individual
requirement.
LEASE DEED
This Lease deed is made and executed at (Name of place) on this ……………….day 20th of
…………, August ………… 2026
BETWEEN
……………………………….,s/od/o………………………………………., Mr. Rajesh Mehta Mr. Mahesh Mehta
r/o…………………….…………………… Kandivali, Mumbai, Maharashtra (hereinafter called the Lessor) of the one part.
AND
……………………………….,s/o Ms. Ananya Sharma d/o………………………………………., Mr. Rakesh Sharma
r/o…………………………………………… Hinjewadi, Pune, Maharashtra (hereinafter called the Lessee) of the other part.
The expression Lessor & Lessee shall mean and include the parties itself, their respective legal
heirs, executors, successors, administrators, legal representatives and assigns/nominees of their
respective part.
Whereas Lessor is an absolute owner and in possession of the property no. 502, ……………., Sunshine Residency
situated at
……………………………………………………measuring…………………….. And

In [6]:
from schema import Contract

In [7]:
print(Contract.schema_json(indent=2))  # Print the schema for the Contract model

{
  "$defs": {
    "Clause": {
      "description": "Represents a clause in a legal document.",
      "properties": {
        "title": {
          "description": "The title of the clause",
          "title": "Title",
          "type": "string"
        },
        "text": {
          "description": "The text content of the clause",
          "title": "Text",
          "type": "string"
        },
        "section": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "description": "The section of the contract where the clause is located (if seperately specified)",
          "title": "Section"
        }
      },
      "required": [
        "title",
        "text"
      ],
      "title": "Clause",
      "type": "object"
    }
  },
  "description": "Represents a legal contract consisting of multiple clauses.",
  "properties": {
    "title": {
      "descri

C:\Users\urja tendolkar\AppData\Local\Temp\ipykernel_48328\50242175.py:1: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(Contract.schema_json(indent=2))  # Print the schema for the Contract model


In [8]:
import os
from langchain_groq import ChatGroq

client = ChatGroq(model="llama-3.3-70b-versatile", 
                  api_key=os.getenv("GROQ_KEY"),
                
)


In [100]:
metadata_start = {
    "lease_deed":["property", "SCHEDULE OF THE LEASED PROPERTY:"],
    "sale_deed":"AGREEMENT TO SELL",
}
conditions_start = {
    "lease_deed": "NOW THIS LEASE DEED WITNESS AS UNDER:-",
    "sale_deed": "9. Conditions of Sale:",
    # add other document types later
}
def extract_clauses(contract, doc_type):
    start_phrase = conditions_start.get(doc_type)
    metadata_phrase = metadata_start.get(doc_type)
    if not start_phrase:
        raise ValueError(f"Unsupported document type: {doc_type}")

    parts = contract.split(start_phrase[0] if isinstance(start_phrase, list) else start_phrase)
    metadata = parts[0].strip() if len(parts) > 1 else ""
    clauses = parts[1].strip() if len(parts) > 1 else contract.strip()
    for phrase in metadata_phrase[1:] if isinstance(metadata_phrase, list) else [metadata_phrase]:
        if clauses.find(phrase) != -1:
            partition = clauses.split(phrase)
            clauses = partition[0].strip()
            metadata += "\n" + partition[1].strip() if len(partition) > 1 else ""

    return metadata, clauses

In [101]:
# @tool
# def identify_unusual_terms(clause):
#     """Identify unusual, restrictive, or potentially concerning terms in a contract clause."""

# @tool
# def summarize_clause(clause):
#     """Explain a contract clause in simple language for a non-legal reader."""

# @tool
# def find_related_clauses(clause: str, contract: str) -> str:
#     """Find other clauses in the contract that may interact with or modify the given clause."""
#     return "..."

# @tool
# def find_contradicting_clauses(clause: str, contract: str) -> str:
#     """Identify other clauses in the contract that contradict or conflict with the given clause."""
#     return "..."

In [105]:
metadata, clauses = extract_clauses(text, "lease_deed")


def cleanup(text):
    # Remove page number + footer
    text = re.sub(
        r'\s*\d+\s*Final model draft agreed to by MoUD and DoLR\n\s*',
        ' ',
        text
    )

    # Remove dotted / ellipsis placeholders
    text = re.sub(
        r'(?:\s*[.…]\s*){2,}',
        ' ',
        text
    )

    # Normalize whitespace and newlines
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Clean metadata
metadata = cleanup(metadata)

# Split clauses by numbered sections
clause_list = re.split(r'\d+\.\s+', clauses)[1:]

# Clean each clause
clause_list = [cleanup(clause) for clause in clause_list]


print("METADATA:")
print(metadata)

print("\nCLAUSES:")
print(clause_list)

METADATA:
Final model draft agreed to by MoUD and DoLR Note: -This is a model draft and may be customized according to individual requirement. LEASE DEED This Lease deed is made and executed at (Name of place) on this day 20th of , August 2026 BETWEEN ,s/od/o , Mr. Rajesh Mehta Mr. Mahesh Mehta r/o Kandivali, Mumbai, Maharashtra (hereinafter called the Lessor) of the one part. AND ,s/o Ms. Ananya Sharma d/o , Mr. Rakesh Sharma r/o Hinjewadi, Pune, Maharashtra (hereinafter called the Lessee) of the other part. The expression Lessor & Lessee shall mean and include the parties itself, their respective legal heirs, executors, successors, administrators, legal representatives and assigns/nominees of their respective part. Whereas Lessor is an absolute owner and in possession of the property no. 502, , Sunshine Residency situated at measuring Andheri West, Mumbai, Maharashtra 1200 sq.ft (hereinafter referred to as the SAID PROPERTY). Revenue District Mumbai Suburban Sub-Registrar Office Andh

In [28]:
@tool
def classify_clause_risk(clause):
    """Classify a contract clause as LOW, MEDIUM, or HIGH risk."""

    prompt = """
    You are a legal document analysis assistant.

    Classify the following contract clause into one of three risk
    categories: LOW, MEDIUM, or HIGH.

    Criteria:

    LOW:
    The clause is standard, commonly accepted, and does not appear
    to impose significant obligations, restrictions, or unusual
    conditions.

    MEDIUM:
    The clause contains obligations, restrictions, ambiguity, or
    potentially burdensome conditions that may require attention
    or clarification.

    HIGH:
    The clause imposes significant obligations, restrictions,
    liabilities, penalties, or other conditions that could have
    serious implications for a party.

    Important:
    - Base the classification only on the provided clause.
    - Do not assume facts that are not present in the clause.
    - Do not declare a clause illegal or legally invalid.
    - Flag potential concerns that may warrant clarification or
      professional legal review.
    - Do not assess contradictions with other clauses here.
      Contradictions are handled separately.

    Respond in the following format:

    clause: <clause text>
    risk_category: <LOW, MEDIUM, or HIGH>
    justification: <brief explanation of the classification>
    """

    response = client.invoke(
        prompt + f"\n\nClause:\n{clause}"
    )

    return response.content

In [29]:
from langchain.agents import create_agent

tools = [
    classify_clause_risk,
]

agent = create_agent(
    model=client,
    tools=tools,
    system_prompt="""
    You are ClauseIQ, a contract analysis assistant.

    Analyze contractual clauses for:
    - Risk classification (LOW, MEDIUM, HIGH)

    Use the available tools when they are useful.
    Do not provide definitive legal advice.
    """
)

In [18]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"""
            Analyze this contract clause:

            {clauses[3]}
            """
        }
    ]
})

In [ ]:
result

{'messages': [HumanMessage(content='\n            Analyze this contract clause:\n\n            \n\n            ', additional_kwargs={}, response_metadata={}, id='41070dac-7d59-4f6a-944d-8b14f788fd48'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'h8gkqwex7', 'function': {'arguments': '{"clause":"the provided clause is empty"}', 'name': 'classify_clause_risk'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 286, 'total_tokens': 307, 'completion_time': 0.069967839, 'completion_tokens_details': None, 'prompt_time': 0.015705515, 'prompt_tokens_details': None, 'queue_time': 0.161891749, 'total_time': 0.085673354}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ffc23-ef37-7010-bacf-9539f8a41c76-0', tool_calls=[{'name': 'classify_clause_risk', 'args': {'clause': 'the pr

: 